In [ ]:
from itertools import product
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
from qiskit_addon_sqd.counts import counts_to_arrays, bit_array_to_arrays,BitArray
from tqdm.notebook import tqdm
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm
from matplotlib import font_manager
import scipy.stats as stats
from shutil import move
font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})

from qiskit_addon_sqd.subsampling import postselect_by_hamming_right_and_left


In [ ]:
datadf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';').dropna(axis=1)

In [ ]:
import math


def fci_dimension(n_alpha: int, n_beta: int, n_orbitals: int) -> int:
    """Calculates the FCI space dimension for a given number of alpha/beta

    electrons and spatial orbitals.
    """
    alpha_combs = math.comb(n_orbitals, n_alpha)
    beta_combs = math.comb(n_orbitals, n_beta)

    return alpha_combs * beta_combs

In [ ]:
dim_data = []
for i in sorted(glob("./counts/*npz")):
    parts = os.path.basename(i).replace('.npz', '').split('_')
    name, _, rawlayers, basis = parts[:4]
    injection = '_'.join(parts[4:])

    n_elec = datadf.loc[datadf['molecule'] == name, 'Ne'].values[0]
    n_orb = datadf.loc[datadf['molecule'] == name, 'No'].values[0]
    right = left = n_elec // 2
    layers = int(rawlayers.strip("L"))
    exact_dim = fci_dimension(left,right,n_orb)
    counts = np.load(i)
    probarr = counts['probarr']
    bitstrings = counts['bitstrings']  # already the right shape/dtype, no conversion needed

    ps_bitstrings, ps_probs = postselect_by_hamming_right_and_left(
        bitstrings, probarr,
        hamming_right=right, hamming_left=left,
    )
    
    dim_data.append((name, layers, basis, injection, ps_bitstrings.shape[0],bitstrings.shape[0],exact_dim,10000))

In [ ]:
pd.DataFrame(dim_data,columns=['Name','L','Basis','Injection','Postselected','Device','Exact','Shots']).to_excel("Dimensions.xlsx")

In [ ]:
datadf